# Realized Volatility Bayesian Prediction

Jakub Kozak, Mateusz Gappa  
Uniwersytet Jagiellonski

This notebook is the clean research version of the volatility project. The goal
is to forecast the mean realized variance over the next 5 trading days using
classical HAR benchmarks and Bayesian HAR models.

## 1. Research Setup

We use daily close prices for the S&P 500 and VIX over the period 2010-01-01 to
2023-01-01.

Daily realized variance is proxied by squared log returns:

$$RV_t = \left(\log\frac{S_t}{S_{t-1}}\right)^2.$$

The target is the average future realized variance:

$$RV^{mean}_t = \frac{1}{5}\sum_{i=1}^{5} RV_{t+i}.$$

The statistical question is whether Bayesian HAR models, especially with a
Student-t likelihood, improve point forecasts, uncertainty quantification, and
high-volatility warning signals relative to simple frequentist benchmarks.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)

import pymc as pm
import arviz as az

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
# Notebook parameters
START = "2010-01-01"
END = "2023-01-01"
TICKERS = ["^GSPC", "^VIX"]
MAIN_ASSET = "SP500"
HORIZON = 5
TRAIN_FRACTION = 0.80
EPS = 1e-12

# Bayesian sampling parameters. Increase for final runs if needed.
DRAWS = 1000
TUNE = 500
CHAINS = 2
TARGET_ACCEPT = 0.90
RANDOM_SEED = 42

## 2. Helper Functions

The functions below keep the notebook readable while staying close to the
statistical workflow. They are intentionally defined here instead of hidden in a
large pipeline.

In [ ]:
def qlike(y_true, y_pred, eps=1e-12):
    """QLIKE loss for variance forecasts. Lower is better."""
    y_true = np.maximum(np.asarray(y_true), eps)
    y_pred = np.maximum(np.asarray(y_pred), eps)
    ratio = y_true / y_pred
    return np.mean(ratio - np.log(ratio) - 1.0)


def evaluate_point_forecast(name, y_true_log, y_pred_log, rv_true):
    """Evaluate a log-RV forecast on both log and variance scales."""
    rv_pred = np.exp(y_pred_log)
    return {
        "model": name,
        "RMSE_log": np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "MAE_log": mean_absolute_error(y_true_log, y_pred_log),
        "R2_log": r2_score(y_true_log, y_pred_log),
        "RMSE_RV": np.sqrt(mean_squared_error(rv_true, rv_pred)),
        "MAE_RV": mean_absolute_error(rv_true, rv_pred),
        "R2_RV": r2_score(rv_true, rv_pred),
        "QLIKE": qlike(rv_true, rv_pred),
    }


def evaluate_posterior_predictive(name, y_true_log, pred_samples_log, rv_true):
    """Evaluate Bayesian posterior predictive samples."""
    pred_mean_log = pred_samples_log.mean(axis=0)

    lower_80_log = np.quantile(pred_samples_log, 0.10, axis=0)
    upper_80_log = np.quantile(pred_samples_log, 0.90, axis=0)
    lower_95_log = np.quantile(pred_samples_log, 0.025, axis=0)
    upper_95_log = np.quantile(pred_samples_log, 0.975, axis=0)

    rv_pred_mean = np.exp(pred_mean_log)
    lower_80_rv = np.exp(lower_80_log)
    upper_80_rv = np.exp(upper_80_log)
    lower_95_rv = np.exp(lower_95_log)
    upper_95_rv = np.exp(upper_95_log)

    result = evaluate_point_forecast(name, y_true_log, pred_mean_log, rv_true)
    result.update({
        "coverage_80_log": np.mean((y_true_log >= lower_80_log) & (y_true_log <= upper_80_log)),
        "coverage_95_log": np.mean((y_true_log >= lower_95_log) & (y_true_log <= upper_95_log)),
        "avg_width_80_log": np.mean(upper_80_log - lower_80_log),
        "avg_width_95_log": np.mean(upper_95_log - lower_95_log),
        "coverage_80_RV": np.mean((rv_true >= lower_80_rv) & (rv_true <= upper_80_rv)),
        "coverage_95_RV": np.mean((rv_true >= lower_95_rv) & (rv_true <= upper_95_rv)),
        "avg_width_80_RV": np.mean(upper_80_rv - lower_80_rv),
        "avg_width_95_RV": np.mean(upper_95_rv - lower_95_rv),
    })

    intervals = {
        "pred_mean_log": pred_mean_log,
        "lower_80_log": lower_80_log,
        "upper_80_log": upper_80_log,
        "lower_95_log": lower_95_log,
        "upper_95_log": upper_95_log,
        "rv_pred_mean": rv_pred_mean,
        "lower_80_rv": lower_80_rv,
        "upper_80_rv": upper_80_rv,
        "lower_95_rv": lower_95_rv,
        "upper_95_rv": upper_95_rv,
    }
    return result, intervals

In [ ]:
def make_har_features(close, horizon=5, eps=1e-12):
    """Build HAR features using only information available at time t."""
    df = close.to_frame("close").copy()
    df["ret_1d"] = np.log(df["close"] / df["close"].shift(1))
    df["RV_1d"] = df["ret_1d"] ** 2

    past_rv = df["RV_1d"].shift(1)
    df["RV_lag_1"] = past_rv
    df["RV_mean_5"] = past_rv.rolling(5).mean()
    df["RV_mean_22"] = past_rv.rolling(22).mean()

    df["RV_forward_5"] = sum(
        df["RV_1d"].shift(-step) for step in range(1, horizon + 1)
    ) / horizon

    df["log_RV_lag_1"] = np.log(df["RV_lag_1"] + eps)
    df["log_RV_mean_5"] = np.log(df["RV_mean_5"] + eps)
    df["log_RV_mean_22"] = np.log(df["RV_mean_22"] + eps)
    df["log_RV_forward_5"] = np.log(df["RV_forward_5"] + eps)

    df["vol_jump_signal"] = df["RV_mean_5"] / (df["RV_mean_22"] + eps)
    df["jump_regime"] = (df["vol_jump_signal"] > 2).astype(int)
    df["log_RV_lag_1_jump"] = df["log_RV_lag_1"] * df["jump_regime"]
    df["log_RV_mean_5_jump"] = df["log_RV_mean_5"] * df["jump_regime"]
    df["log_RV_mean_22_jump"] = df["log_RV_mean_22"] * df["jump_regime"]

    return df.dropna()


def chronological_split(model_df, train_fraction=0.8):
    split_idx = int(len(model_df) * train_fraction)
    return model_df.iloc[:split_idx].copy(), model_df.iloc[split_idx:].copy()

In [ ]:
def fit_bayesian_har(
    X_train,
    y_train,
    X_test,
    feature_names,
    likelihood="student",
    draws=1000,
    tune=500,
    chains=2,
    target_accept=0.9,
    random_seed=42,
):
    """Fit Bayesian HAR and simulate out-of-sample posterior predictions."""
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test)
    y_train_scaled = y_scaler.fit_transform(np.asarray(y_train).reshape(-1, 1)).ravel()

    coords = {"obs_id": np.arange(X_train_scaled.shape[0]), "feature": feature_names}

    with pm.Model(coords=coords) as model:
        X_data = pm.Data("X_data", X_train_scaled, dims=("obs_id", "feature"))
        y_data = pm.Data("y_data", y_train_scaled, dims="obs_id")

        alpha = pm.Normal("alpha", mu=0.0, sigma=1.0)
        beta = pm.Normal("beta", mu=0.0, sigma=1.0, dims="feature")
        sigma = pm.HalfNormal("sigma", sigma=1.0)
        mu = pm.Deterministic("mu", alpha + pm.math.dot(X_data, beta), dims="obs_id")

        if likelihood == "normal":
            pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_data, dims="obs_id")
            model_name = "Bayesian HAR Normal"
        elif likelihood == "student":
            nu_minus_two = pm.Exponential("nu_minus_two", lam=1 / 10)
            nu = pm.Deterministic("nu", nu_minus_two + 2)
            pm.StudentT("y_obs", nu=nu, mu=mu, sigma=sigma, observed=y_data, dims="obs_id")
            model_name = "Bayesian HAR Student-t"
        else:
            raise ValueError("likelihood must be 'normal' or 'student'")

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
        )

    posterior = idata.posterior.stack(sample=("chain", "draw"))
    alpha_samples = posterior["alpha"].values
    beta_samples = posterior["beta"].transpose("sample", "feature").values
    sigma_samples = posterior["sigma"].values

    mu_test = alpha_samples[:, None] + beta_samples @ X_test_scaled.T
    rng = np.random.default_rng(random_seed)

    if likelihood == "normal":
        pred_scaled = rng.normal(mu_test, sigma_samples[:, None])
    else:
        nu_samples = posterior["nu"].values
        pred_scaled = mu_test + sigma_samples[:, None] * rng.standard_t(
            df=nu_samples[:, None],
            size=mu_test.shape,
        )

    pred_log = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
    return model_name, idata, pred_log

## 3. Data Download

The presentation uses S&P 500 as the main asset and VIX as market-volatility
context. The core model below starts with S&P 500. VIX is loaded so it can be
added as an exogenous predictor later.

In [ ]:
# Put yfinance cache inside the project when the installed version supports it.
try:
    import yfinance.cache as yf_cache
    yf_cache.set_cache_location("data/yfinance_cache")
except Exception:
    pass

raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
close = raw["Close"].rename(columns={"^GSPC": "SP500", "^VIX": "VIX"})
close.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
close["SP500"].plot(ax=ax, label="S&P 500")
ax.set_title("S&P 500 close price")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
plt.show()

## 4. HAR Feature Construction

The HAR idea is that volatility has heterogeneous memory: yesterday, last week,
and last month can all matter.

We use:

- `RV_lag_1`: yesterday's realized variance
- `RV_mean_5`: trailing 5-day realized variance
- `RV_mean_22`: trailing 22-day realized variance

For the main statistical model we work on the log scale.

In [ ]:
model_df = make_har_features(close[MAIN_ASSET], horizon=HORIZON, eps=EPS)
train_df, test_df = chronological_split(model_df, TRAIN_FRACTION)

features = ["log_RV_lag_1", "log_RV_mean_5", "log_RV_mean_22"]
target = "log_RV_forward_5"

X_train = train_df[features]
X_test = test_df[features]
y_train = train_df[target]
y_test = test_df[target]
rv_true = test_df["RV_forward_5"].values

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
model_df[["RV_forward_5"] + features].head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
model_df["RV_forward_5"].plot(ax=ax, label="5-day forward RV")
ax.set_title("Target: 5-day forward realized variance")
ax.set_xlabel("Date")
ax.set_ylabel("RV")
ax.legend()
plt.show()

## 5. Classical HAR Benchmarks

We first fit OLS and Ridge HAR models on `log(RV_forward_5)`. These are not the
final model; they are transparent benchmarks.

In [ ]:
ols = LinearRegression()
ols.fit(X_train, y_train)
y_pred_log_ols = ols.predict(X_test)

ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])
ridge.fit(X_train, y_train)
y_pred_log_ridge = ridge.predict(X_test)

baseline_results = pd.DataFrame([
    evaluate_point_forecast("OLS HAR log(RV)", y_test.values, y_pred_log_ols, rv_true),
    evaluate_point_forecast("Ridge HAR log(RV)", y_test.values, y_pred_log_ridge, rv_true),
])
baseline_results

In [ ]:
coef_df = pd.DataFrame({
    "feature": features,
    "OLS_coefficient": ols.coef_,
})
print("OLS intercept:", ols.intercept_)
coef_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values, label="True log(RV_forward_5)", linewidth=1)
ax.plot(y_test.index, y_pred_log_ols, label="OLS HAR", linewidth=1)
ax.plot(y_test.index, y_pred_log_ridge, label="Ridge HAR", linewidth=1)
ax.set_title("Classical HAR forecasts on log realized variance")
ax.set_xlabel("Date")
ax.set_ylabel("log(RV)")
ax.legend()
plt.show()

## 6. Jump-Regime HAR Extension

This matches the presentation's jump-signal idea. It is kept as a diagnostic
extension, not as the main final model.

The signal is:

$$JS_t = RV^{-5}_t / RV^{-22}_t,$$

and the jump regime is active when `JS_t > 2`.

In [ ]:
jump_features = features + [
    "vol_jump_signal",
    "jump_regime",
    "log_RV_lag_1_jump",
    "log_RV_mean_5_jump",
    "log_RV_mean_22_jump",
]

X_train_jump = train_df[jump_features]
X_test_jump = test_df[jump_features]

jump_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])
jump_ridge.fit(X_train_jump, y_train)
y_pred_log_jump = jump_ridge.predict(X_test_jump)

jump_result = pd.DataFrame([
    evaluate_point_forecast("Jump-regime Ridge HAR", y_test.values, y_pred_log_jump, rv_true)
])
jump_result

## 7. Correlation and Naive Forecasts

Before Bayesian modeling, check whether the HAR features genuinely carry signal
and whether the model beats naive lagged-volatility predictors.

In [ ]:
corr = model_df[[target] + features].corr()
corr

In [ ]:
naive_predictions = {
    "Naive log_RV_lag_1": X_test["log_RV_lag_1"].values,
    "Naive log_RV_mean_5": X_test["log_RV_mean_5"].values,
    "Naive log_RV_mean_22": X_test["log_RV_mean_22"].values,
}

naive_results = pd.DataFrame([
    evaluate_point_forecast(name, y_test.values, pred, rv_true)
    for name, pred in naive_predictions.items()
])
naive_results

## 8. Bayesian HAR Models

We now fit two Bayesian versions of the log-HAR regression:

1. Normal likelihood
2. Student-t likelihood

The Student-t model is the more important one statistically because volatility
forecast residuals tend to have heavy tails and crisis spikes.

In [ ]:
normal_name, normal_idata, normal_pred_log = fit_bayesian_har(
    X_train,
    y_train,
    X_test,
    features,
    likelihood="normal",
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=TARGET_ACCEPT,
    random_seed=RANDOM_SEED,
)

student_name, student_idata, student_pred_log = fit_bayesian_har(
    X_train,
    y_train,
    X_test,
    features,
    likelihood="student",
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=TARGET_ACCEPT,
    random_seed=RANDOM_SEED,
)

In [ ]:
normal_result, normal_intervals = evaluate_posterior_predictive(
    normal_name,
    y_test.values,
    normal_pred_log,
    rv_true,
)

student_result, student_intervals = evaluate_posterior_predictive(
    student_name,
    y_test.values,
    student_pred_log,
    rv_true,
)

all_results = pd.DataFrame(
    baseline_results.to_dict("records")
    + jump_result.to_dict("records")
    + [normal_result, student_result]
)
all_results

In [ ]:
az.summary(normal_idata, var_names=["alpha", "beta", "sigma"])

In [ ]:
az.summary(student_idata, var_names=["alpha", "beta", "sigma", "nu"])

## 9. Posterior Predictive Intervals

Point forecasts are not enough. For volatility, interval calibration matters:
a useful Bayesian model should say when uncertainty is wide.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values, label="True log(RV_forward_5)", linewidth=1)
ax.plot(y_test.index, normal_intervals["pred_mean_log"], label="Bayesian Normal mean", linewidth=1)
ax.fill_between(
    y_test.index,
    normal_intervals["lower_80_log"],
    normal_intervals["upper_80_log"],
    alpha=0.25,
    label="80% interval",
)
ax.fill_between(
    y_test.index,
    normal_intervals["lower_95_log"],
    normal_intervals["upper_95_log"],
    alpha=0.12,
    label="95% interval",
)
ax.set_title("Bayesian HAR Normal: posterior predictive intervals")
ax.set_xlabel("Date")
ax.set_ylabel("log(RV)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values, label="True log(RV_forward_5)", linewidth=1)
ax.plot(y_test.index, student_intervals["pred_mean_log"], label="Bayesian Student-t mean", linewidth=1)
ax.fill_between(
    y_test.index,
    student_intervals["lower_80_log"],
    student_intervals["upper_80_log"],
    alpha=0.25,
    label="80% interval",
)
ax.fill_between(
    y_test.index,
    student_intervals["lower_95_log"],
    student_intervals["upper_95_log"],
    alpha=0.12,
    label="95% interval",
)
ax.set_title("Bayesian HAR Student-t: posterior predictive intervals")
ax.set_xlabel("Date")
ax.set_ylabel("log(RV)")
ax.legend()
plt.show()

## 10. Residual Diagnostics

The Student-t likelihood is motivated by the fact that OLS residuals are often
heavy-tailed during stress periods.

In [ ]:
ols_residuals = y_test.values - y_pred_log_ols

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_test.index, ols_residuals)
axes[0].axhline(0, linestyle="--", color="black", linewidth=1)
axes[0].set_title("OLS HAR residuals")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual")

axes[1].hist(ols_residuals, bins=50)
axes[1].set_title("Histogram of OLS residuals")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 11. High-Volatility Warning Signals

Following the presentation, define a high-volatility day using the 80th
percentile of realized `RV_forward_5`. Then compare warning signals based on
mean forecasts and Bayesian upper predictive bounds.

In [ ]:
def evaluate_warning(name, y_true, y_signal):
    return {
        "signal": name,
        "precision": precision_score(y_true, y_signal, zero_division=0),
        "recall": recall_score(y_true, y_signal, zero_division=0),
        "f1": f1_score(y_true, y_signal, zero_division=0),
        "n_warnings": int(y_signal.sum()),
        "n_high_vol": int(y_true.sum()),
    }

threshold = model_df["RV_forward_5"].quantile(0.80)
high_vol_realized = (rv_true > threshold).astype(int)

warning_signals = {
    "OLS mean forecast": np.exp(y_pred_log_ols),
    "Bayesian Student-t mean forecast": student_intervals["rv_pred_mean"],
    "Bayesian Student-t upper 80%": student_intervals["upper_80_rv"],
    "Bayesian Student-t upper 95%": student_intervals["upper_95_rv"],
}

warning_results = pd.DataFrame([
    evaluate_warning(name, high_vol_realized, (values > threshold).astype(int))
    for name, values in warning_signals.items()
])
warning_results

## 12. Main Takeaways

Use this final cell for the written interpretation after running the notebook.
The expected discussion should cover:

- whether Student-t improves QLIKE relative to OLS/Ridge,
- whether posterior intervals are reasonably calibrated,
- whether warning signals trade precision for recall,
- whether jump/regime features help or overfit,
- whether VIX should be added as an exogenous predictor in the next version.